# Домашнее задание: Retrieval-Augmented Generation (RAG)

**Курс:** NLP-2  
**Тема:** Построение, оптимизация и комплексная оценка RAG-систем.

## Введение
В рамках данного задания вам предстоит пройти полный путь ML-инженера при работе с RAG: от сборки базового пайплайна до тонкой настройки ретривера и генератора. Мы будем работать с датасетом **SciFact** (проверка научных фактов).

### Методология Эксперимента
Важно соблюдать гигиену ML-экспериментов. Мы разделили данные на два сета:
1.  **Main Split (200 запросов):** Ваша "Dev" выборка. Все промежуточные прогоны, подбор гиперпараметров (chunk size, top_k) и отладку вы делаете **только** на ней. Мы хотим избежать переобучения под тестовые данные.
2.  **Challenge Split (50 запросов):** Ваша "Test" выборка. Вы используете её **ровно один раз** в самом конце ноутбука для финальной валидации лучшей конфигурации.

---

## 1. Подготовка окружения
Установка зависимостей. Мы используем `LlamaIndex` как оркестратор, `Qdrant` как векторную БД и `HuggingFace` для моделей.

In [3]:
%%capture
%pip install llama-index-core llama-index-llms-huggingface llama-index-embeddings-huggingface llama-index-vector-stores-qdrant llama-index-retrievers-bm25
%pip install qdrant-client ir_datasets ir_measures bitsandbytes accelerate transformers
%pip install pandas matplotlib seaborn tqdm

In [1]:
# JAX Memory Configuration (CRITICAL: must run before imports)
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
print("JAX configured")

JAX configured


In [2]:
import os
import time
import random
import gc
from dataclasses import dataclass, field
from typing import List, Optional, Dict

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# Optimizations for T4 GPU in Colab
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ARTIFACTS_DIR = "/content/artifacts"
QDRANT_PATH = f"{ARTIFACTS_DIR}/qdrant_local"
LOGS_DIR = f"{ARTIFACTS_DIR}/logs"

os.makedirs(QDRANT_PATH, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

c:\Users\Gulfik\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Загрузка данных (SciFact)
Загружаем корпус и запросы. SciFact — это сложный датасет, где запрос часто требует понимания причинно-следственных связей в научных текстах.

In [3]:
import ir_datasets

ds_corpus = ir_datasets.load("beir/scifact")
ds_test = ir_datasets.load("beir/scifact/test")

print("Loading corpus...")
corpus_rows = [
    {"doc_id": str(d.doc_id), "title": d.title or "", "text": d.text or ""}
    for d in ds_corpus.docs_iter()
]
df_corpus = pd.DataFrame(corpus_rows)

print("Loading queries...")
query_rows = [
    {"query_id": str(q.query_id), "text": q.text} for q in ds_test.queries_iter()
]
df_queries = pd.DataFrame(query_rows)

print("Loading qrels...")
qrel_rows = [
    {"query_id": str(q.query_id), "corpus_id": str(q.doc_id), "score": int(q.relevance)}
    for q in ds_test.qrels_iter()
]
df_qrels = pd.DataFrame(qrel_rows)

# Map for metrics calculation
qrels_map = (
    df_qrels.groupby("query_id")
    .apply(lambda x: dict(zip(x["corpus_id"], x["score"])))
    .to_dict()
)

# Fixed Splits
all_qids = df_queries["query_id"].unique()
rng = np.random.default_rng(42)
rng.shuffle(all_qids)
split_main = all_qids[:200]
split_challenge = all_qids[200:250]

print(f"Corpus size: {len(df_corpus)}")
print(
    f"Main split: {len(split_main)} queries, Challenge split: {len(split_challenge)} queries"
)

Loading corpus...
Loading queries...
Loading qrels...
Corpus size: 5183
Main split: 200 queries, Challenge split: 50 queries


C:\Temp\ipykernel_18592\4130493645.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: dict(zip(x["corpus_id"], x["score"])))


## 3. Конфигурация и Модели
Мы используем `dataclasses` для строгой типизации конфигов экспериментов.

In [4]:
from llama_index.core import VectorStoreIndex, Document, Settings, StorageContext
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core.node_parser import SentenceSplitter, HierarchicalNodeParser
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from qdrant_client import QdrantClient
from transformers import BitsAndBytesConfig, AutoTokenizer


# --- CONFIG CLASSES ---
@dataclass
class ChunkingConfig:
    chunk_size: int = 512
    chunk_overlap: int = 100
    use_hierarchical: bool = False
    child_chunk_size: int = 128


@dataclass
class RetrievalConfig:
    top_k: int = 5  # Final k documents for LLM
    overfetch_k: int = 15  # Candidates fetched from DB (The "Funnel" top)
    mode: str = "dense"  # "dense" | "hybrid"
    use_reranker: bool = False
    rerank_top_n: int = 5  # Documents to keep after reranking


@dataclass
class LLMConfig:
    model_name: str = "Qwen/Qwen3-4B-Instruct-2507"
    max_new_tokens: int = 256
    context_window: int = 2048
    load_in_4bit: bool = True
    # Subsample for E2E evaluation speed
    e2e_eval_n: int = 10
    prompt_template: str = "Context:\n{context_str}\n\nQuery: {query_str}\nAnswer:"


@dataclass
class EmbeddingConfig:
    model_name: str = "Qwen/Qwen3-Embedding-0.6B"
    truncate_dim: Optional[int] = None  # For Matryoshka learning


@dataclass
class RAGConfig:
    name: str = "baseline"
    chunking: ChunkingConfig = field(default_factory=ChunkingConfig)
    retrieval: RetrievalConfig = field(default_factory=RetrievalConfig)
    llm: LLMConfig = field(default_factory=LLMConfig)
    embedding: EmbeddingConfig = field(default_factory=EmbeddingConfig)
    qdrant_collection: str = "scifact_dense_base"
    recreate_collection: bool = False
    rerank_model: str = "Qwen/Qwen3-Reranker-0.6B"


# --- MODEL HELPERS ---
_CACHED_LLM = None
_CACHED_EMBED = None


def unload_embedder():
    global _CACHED_EMBED
    if _CACHED_EMBED:
        del _CACHED_EMBED
        _CACHED_EMBED = None
        Settings.embed_model = None
        gc.collect()
        torch.cuda.empty_cache()


def get_embedder(cfg: EmbeddingConfig):
    global _CACHED_EMBED
    if _CACHED_EMBED:
        return _CACHED_EMBED
    _CACHED_EMBED = HuggingFaceEmbedding(
        model_name=cfg.model_name,
        device="cuda",
        normalize=True,
        truncate_dim=cfg.truncate_dim,
    )
    return _CACHED_EMBED


def get_llm(cfg: LLMConfig):
    global _CACHED_LLM
    if _CACHED_LLM:
        return _CACHED_LLM
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
    tokenizer = AutoTokenizer.from_pretrained(
        cfg.model_name,
        trust_remote_code=True,
        use_fast=False,
    )
    _CACHED_LLM = HuggingFaceLLM(
        model_name=cfg.model_name,
        context_window=cfg.context_window,
        max_new_tokens=cfg.max_new_tokens,
        model_kwargs={"quantization_config": bnb},
        generate_kwargs={"do_sample": False},
        device_map="auto",
        tokenizer=tokenizer,
    )
    return _CACHED_LLM

resource module not available on Windows


## 4. Система Метрик (Metrics System)

Для комплексной оценки качества мы используем двухуровневую систему метрик.

### 1. Retrieval Phase (Поиск)
Оценивает, насколько релевантные куски текста (chunks) мы нашли в базе.

**A. С учителем (Supervised / Reference-based):**
Требуют наличия "правильных ответов" (qrels).
*   **`nDCG@k`**: Основная метрика ранжирования. Учитывает не только факт нахождения правильного документа, но и его позицию (чем выше, тем лучше).
*   **`Recall@k`**: Полнота. Какую долю всех релевантных документов мы нашли в топ-k.

**B. Без учителя (Unsupervised / Reference-free):**
Полезны для мониторинга в продакшене, где нет разметки.
*   **`Mean Relevance Score`**: Среднее значение скора (similarity), который выдает ретривер. Показывает "уверенность" модели в найденном.
*   **`Redundancy`**: Избыточность. Показывает, насколько найденные чанки похожи друг на друга. Высокая избыточность — плохо (мы хотим разнообразный контекст).

### 2. Generation Phase (Генерация)
Оценивает качество финального ответа с использованием **LLM-as-a-Judge** (одна LLM оценивает другую).

*   **`Faithfulness` (Answer vs Context):** Верность контексту. Проверяет, что ответ сгенерирован **исключительно** на основе найденных документов, без галлюцинаций.
*   **`Answer Relevancy` (Answer vs Query):** Релевантность вопроса. Проверяет, что сгенерированный текст действительно отвечает на поставленный вопрос пользователя.

In [5]:
from typing import Callable, Any, TypeAlias

BuildPipeline: TypeAlias = Callable[[RAGConfig, pd.DataFrame, bool], Any] | None

In [6]:
import ir_measures
from ir_measures import nDCG, Recall, RR


RUNS_CSV_PATH = os.path.join(LOGS_DIR, "runs.csv")


def compute_redundancy(contexts: List[str]) -> float:
    """Calculates semantic redundancy (higher = more duplicate info)."""
    if not contexts:
        return 0.0
    unique = len(set(contexts))
    return 1.0 - (unique / len(contexts))


def compute_retrieval_metrics(qrels_map, run_doc_ids, run_contexts, run_scores, k=5):
    # 1. Reference-based Metrics (Ground Truth needed)
    run_ir = {}
    for qid, doc_ids in run_doc_ids.items():
        run_ir[qid] = {
            doc_id: float(-(rank + 1)) for rank, doc_id in enumerate(doc_ids[:k])
        }

    measures = [nDCG @ k, Recall @ k]
    agg = ir_measures.calc_aggregate(measures, qrels_map, run_ir)

    # 2. Reference-free Metrics (No Ground Truth)
    red_vals = [compute_redundancy(ctxs[:k]) for ctxs in run_contexts.values()]
    # Mean Score: How confident is the retriever?
    score_vals = [np.mean(scores[:k]) for scores in run_scores.values() if scores]

    return {
        "ndcg@k": float(agg[nDCG @ k]),
        "recall@k": float(agg[Recall @ k]),
        "ref_free_mean_score": float(np.mean(score_vals)) if score_vals else 0.0,
        "ref_free_redundancy": float(np.mean(red_vals)) if red_vals else 0.0,
    }


def log_run(cfg, split, stage, run_name, metrics):
    row = {
        "ts": time.strftime("%H:%M:%S", time.gmtime()),
        "run_name": run_name,
        "split": split,
        "stage": stage,
        **metrics,
    }
    df_old = (
        pd.read_csv(RUNS_CSV_PATH) if os.path.exists(RUNS_CSV_PATH) else pd.DataFrame()
    )
    pd.concat([df_old, pd.DataFrame([row])], ignore_index=True).to_csv(
        RUNS_CSV_PATH, index=False
    )


def show_leaderboard(stage="retrieval", split="main"):
    if not os.path.exists(RUNS_CSV_PATH):
        return
    df = pd.read_csv(RUNS_CSV_PATH)
    df_sub = df[(df["stage"] == stage) & (df["split"] == split)].copy()
    if df_sub.empty:
        return

    # Sort by key metric
    sort_col = "ndcg@k" if stage == "retrieval" else "faithfulness"
    if sort_col in df_sub.columns:
        df_sub = df_sub.sort_values(sort_col, ascending=False)

    df_sub = (
        df_sub.sort_values(sort_col, ascending=False)
        .groupby("run_name", as_index=False)
        .first()
    )

    # Select columns to display
    cols_ret = [
        "run_name",
        "ndcg@k",
        "recall@k",
        "ref_free_mean_score",
        "ref_free_redundancy",
        "latency_s",
    ]
    cols_gen = ["run_name", "faithfulness", "answer_relevancy", "latency_s"]

    cols = cols_ret if stage == "retrieval" else cols_gen
    print(f"\n🏆 LEADERBOARD [{stage.upper()} | {split.upper()}] 🏆")
    display(df_sub[[c for c in cols if c in df_sub.columns]])


def _get_df(split):
    return df_queries[
        df_queries["query_id"].isin(split_main if split == "main" else split_challenge)
    ]


def run_retrieval(
    cfg: RAGConfig,
    split: str,
    run_name: str,
    build_pipeline: BuildPipeline = None,
):
    unload_embedder()
    
    index, retriever, _ = (
        build_rag_pipeline(cfg, df_corpus, load_llm=False)
        if build_pipeline is None
        else build_pipeline(cfg, df_corpus, False)
    )

    df_q = _get_df(split)
    results_dids, results_txts, results_scores = {}, {}, {}

    t0 = time.time()
    for row in tqdm(
        df_q.itertuples(index=False), total=len(df_q), desc=f"Retr {run_name}"
    ):
        nodes = retriever.retrieve(row.text)
        # De-duplicate by doc_id to avoid metrics skew
        seen, dids, txts, scores = set(), [], [], []
        for n in nodes:
            if n.metadata["doc_id"] not in seen:
                seen.add(n.metadata["doc_id"])
                dids.append(n.metadata["doc_id"])
                txts.append(n.get_text())
                scores.append(n.score if n.score else 0.0)
        results_dids[str(row.query_id)] = dids
        results_txts[str(row.query_id)] = txts
        results_scores[str(row.query_id)] = scores

    # FIX: filter qrels_map based on current split
    split_qids = set(df_q["query_id"].astype(str))
    qrels_map_filtered = {
        qid: docs for qid, docs in qrels_map.items() if qid in split_qids
    }

    latency = time.time() - t0

    metrics = compute_retrieval_metrics(
        qrels_map_filtered,
        results_dids,
        results_txts,
        results_scores,
        k=cfg.retrieval.top_k,
    )
    metrics["latency_s"] = latency
    log_run(cfg, split, "retrieval", run_name, metrics)
    # show_leaderboard("retrieval", split)
    unload_embedder()


def run_generation(
    cfg: RAGConfig,
    split: str,
    run_name: str,
    build_pipeline: BuildPipeline = None,
):
    """Generates answers and saves them to JSON for subsequent evaluation."""
    # 1. Retrieval Phase
    unload_embedder()
    index, retriever, _ = (
        build_rag_pipeline(cfg, df_corpus, load_llm=False)
        if build_pipeline is None
        else build_pipeline(cfg, df_corpus, False)
    )

    df_q = _get_df(split).head(cfg.llm.e2e_eval_n)
    records = []
    for row in tqdm(
        df_q.itertuples(index=False), total=len(df_q), desc="Retrieving Contexts"
    ):
        nodes = retriever.retrieve(row.text)
        ctx_list = [n.get_text() for n in nodes[: cfg.retrieval.top_k]]
        records.append(
            {"query_id": str(row.query_id), "q": row.text, "ctx": "\n".join(ctx_list)}
        )

    unload_embedder()

    # 2. Generation Phase
    # Settings.embed_model = None
    Settings.embed_model = get_embedder(cfg.embedding) # необходима инициализация для прогона на тесте
    llm = get_llm(cfg.llm)
    Settings.llm = llm
    print("LLM type:", type(Settings.llm))

    t0 = time.time()
    for item in tqdm(records, desc="Generating"):
        prompt = cfg.llm.prompt_template.format(
            context_str=item["ctx"], query_str=item["q"]
        )
        try:
            item["a"] = llm.complete(prompt).text
        except:
            item["a"] = ""
    latency = time.time() - t0

    # 3. Save results
    output_path = os.path.join(LOGS_DIR, f"generations_{run_name}_{split}.json")
    import json

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "records": records,
                "latency_s": latency,
                "cfg_name": cfg.name,
                "run_name": run_name,
                "split": split,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    print(f"✅ Generations saved to {output_path}")
    return output_path


def run_evaluation(generation_file: str, eval_name: Optional[str] = None):
    """Evaluates already generated answers."""
    import json

    # Load generated data
    with open(generation_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    records = data["records"]
    latency = data["latency_s"]
    split = data.get("split", "main")
    run_name = eval_name if eval_name else data.get("run_name", "eval")

    # Load LLM for Judge
    # Settings.embed_model = None
    Settings.embed_model = get_embedder(RAGConfig().embedding) # необходима инициализация для прогона на тесте
    llm = get_llm(LLMConfig())
    Settings.llm = llm

    # Judge Phase
    faith_scores = []
    rel_scores = []

    for item in tqdm(records, desc=f"Judging {run_name}"):
        if not item.get("a", ""):
            faith_scores.append(0.0)
            rel_scores.append(0.0)
            continue

        # Metric 1: Faithfulness (Context vs Answer)
        p_faith = f"Context: {item['ctx'][:1000]}\nAnswer: {item['a']}\nDoes the Answer use the Context? YES/NO."
        res_f = llm.complete(p_faith, max_new_tokens=5).text.upper()
        faith_scores.append(1.0 if "YES" in res_f else 0.0)

        # Metric 2: Answer Relevancy (Question vs Answer)
        p_rel = f"Question: {item['q']}\nAnswer: {item['a']}\nDoes the Answer address the Question? YES/NO."
        res_r = llm.complete(p_rel, max_new_tokens=5).text.upper()
        rel_scores.append(1.0 if "YES" in res_r else 0.0)

    metrics = {
        "faithfulness": np.mean(faith_scores),
        "answer_relevancy": np.mean(rel_scores),
        "latency_s": latency,
    }

    log_run(RAGConfig(name=run_name), split, "e2e", run_name, metrics)
    # show_leaderboard("e2e", split)
    return metrics


def run_e2e(
    cfg: RAGConfig, split: str, run_name: str, generation_file: Optional[str] = None
):
    """
    End-to-end evaluation with option to reuse generations.

    Args:
        cfg: RAG configuration
        split: "main" or "challenge"
        run_name: Experiment name
        generation_file: Path to previously generated answers file (optional)
    """
    if generation_file and os.path.exists(generation_file):
        # Reuse existing generation
        print(f"📂 Loading generations from {generation_file}")
        return run_evaluation(generation_file, run_name)
    else:
        # Full cycle: generation + evaluation
        gen_file = run_generation(cfg, split, run_name)
        return run_evaluation(gen_file, run_name)


## Task 0: Реализация Baseline Pipeline (4 балла)

Ваша первая задача — реализовать функцию `build_rag_pipeline`. На данном этапе требуется собрать простую архитектуру Dense Retrieval.

**Требования к реализации:**
1.  Инициализация моделей через `Settings` (используйте `get_embedder` и `get_llm`).
2.  Подключение к `QdrantVectorStore`.
3.  Логика индексации: Если коллекция не существует (или `recreate_collection=True`), создать её из `df_corpus`, используя `SentenceSplitter`. Если существует — загрузить существующий индекс.
4.  Возврат объектов `index`, `retriever` и `query_engine`.

Обратите внимание на параметр `overfetch_k` в конфиге. Ретривер должен извлекать именно это количество кандидатов.

In [7]:
def df_to_documents(df: pd.DataFrame) -> list[Document]:
    documents = []

    for _, row in df.iterrows():
        metadata = {
            "doc_id": row["doc_id"],
            "title": row["title"],
        }
        document = Document(text=row["text"], metadata=metadata)
        documents.append(document)

    return documents

In [8]:
_QDRANT_CLIENT = None


def get_qdrant_client():
    """
    Создаем один клиент, чтобы не закрывать каждый раз клиент
    и не получать AlreadyLocked: Permission denied
    """
    global _QDRANT_CLIENT
    if _QDRANT_CLIENT is None:
        _QDRANT_CLIENT = QdrantClient(path=QDRANT_PATH)
    return _QDRANT_CLIENT

In [9]:
def build_rag_pipeline(cfg: RAGConfig, df_corpus: pd.DataFrame, load_llm: bool = True):
    # 1. Setup Models
    Settings.embed_model = get_embedder(cfg.embedding)
    Settings.llm = get_llm(cfg.llm)
    # if load_llm:
    #     Settings.llm = get_llm(cfg.llm)
    # else:
        # Settings.llm = None

    # 2. Vector Store Setup (TODO)
    client = get_qdrant_client()
    vector_store = QdrantVectorStore(
        collection_name=cfg.qdrant_collection, client=client
    )
    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    # 3. Indexing Logic (TODO)
    # Check if collection exists.
    # If yes & not recreate -> load index.
    # Else -> create documents, setup splitter, build index.
    # print(client.collection_exists(collection_name=cfg.qdrant_collection))
    is_collection_exist = client.collection_exists(
        collection_name=cfg.qdrant_collection
    )
    print(f"Collection exist: {is_collection_exist}")

    if is_collection_exist and not cfg.recreate_collection:
        index = VectorStoreIndex.from_vector_store(
            vector_store=vector_store,
            storage_context=storage_context,
            show_progress=True,
        )
    else:
        if cfg.chunking.use_hierarchical:
            parser = HierarchicalNodeParser.from_defaults(
                chunk_sizes=[cfg.chunking.chunk_size, cfg.chunking.child_chunk_size],
                chunk_overlap=cfg.chunking.chunk_overlap,
            )
            transformations = [parser]
        else:
            splitter = SentenceSplitter(
                chunk_size=cfg.chunking.chunk_size,
                chunk_overlap=cfg.chunking.chunk_overlap,
            )
            transformations = [splitter]

        client.delete_collection(collection_name=cfg.qdrant_collection)
        splitter = SentenceSplitter(
            chunk_size=cfg.chunking.chunk_size,
            chunk_overlap=cfg.chunking.chunk_overlap,
        )
        index = VectorStoreIndex.from_documents(
            documents=df_to_documents(df_corpus),
            storage_context=storage_context,
            transformations=transformations,
            show_progress=True,
        )

    # 4. Retrieval Construction (TODO)
    # retriever = ... (use cfg.retrieval.overfetch_k)
    retriever = index.as_retriever(similarity_top_k=cfg.retrieval.overfetch_k)
    # query_engine = ...
    query_engine = index.as_query_engine(similarity_top_k=cfg.retrieval.top_k)
    # <YOUR CODE HERE>

    return index, retriever, query_engine

In [9]:
# TEST YOUR BASELINE
baseline_cfg = RAGConfig(
    name="baseline",
    chunking=ChunkingConfig(chunk_size=512, chunk_overlap=50),
    retrieval=RetrievalConfig(top_k=5, overfetch_k=15),
    qdrant_collection="scifact_base_512",
)

print("Running Baseline...")
# 1. Run Retrieval
# TODO: Uncomment after implementing pipeline
run_retrieval(baseline_cfg, "main", "baseline")

# 2. Check Results
show_leaderboard("retrieval", "main")

Running Baseline...


c:\Users\Gulfik\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\module.py:1357: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:35.)
  return t.to(


LLM is explicitly disabled. Using MockLLM.
Collection exist: True


Retr baseline: 100%|██████████| 200/200 [00:19<00:00, 10.10it/s]


🏆 LEADERBOARD [RETRIEVAL | MAIN] 🏆


,run_name,ndcg@k,recall@k,ref_free_mean_score,ref_free_redundancy,latency_s
1,baseline_256,0.679200,0.76575,0.582125,0.0,30.382416
0,baseline,0.673754,0.76325,0.572684,0.0,19.476203
2,baseline,0.673754,0.76325,0.572684,0.0,19.807326


Embeddings have been explicitly disabled. Using MockEmbedding.


## Task 1: Chunking Strategies (2 балла)

Размер контекста критически влияет на качество поиска.

**Задание:**
1.  **Small vs Large:** Создайте конфигурации для `chunk_size=256` и `chunk_size=1024`. Запустите эксперименты. *Напоминание: при изменении чанкинга необходимо менять имя коллекции или ставить `recreate_collection=True`.*
2.  **Hierarchical Chunking:** Реализуйте иерархический чанкинг (`use_hierarchical=True`). Для этого в `build_rag_pipeline` необходимо добавить проверку конфига и использование `HierarchicalNodeParser`. Этот метод индексирует мелкие чанки, но возвращает контекст родительских (крупных) блоков.

In [10]:
# TODO: Update build_rag_pipeline to support HierarchicalNodeParser

# TODO: Define Configs
cfg_256 = RAGConfig(
    name="baseline_256",
    chunking=ChunkingConfig(chunk_size=256, chunk_overlap=50),
    retrieval=RetrievalConfig(top_k=5, overfetch_k=15),
    qdrant_collection="scifact_base_256",
    # recreate_collection=True,
)

cfg_1024 = RAGConfig(
    name="baseline_1024",
    chunking=ChunkingConfig(chunk_size=1024, chunk_overlap=50),
    retrieval=RetrievalConfig(top_k=5, overfetch_k=15),
    qdrant_collection="scifact_base_1024",
    # recreate_collection=True,
)

cfg_hier = RAGConfig(
    name="baseline_hier",
    chunking=ChunkingConfig(use_hierarchical=True),
    retrieval=RetrievalConfig(top_k=5, overfetch_k=15),
    qdrant_collection="scifact_base_hier_512_128",
    # recreate_collection=True,
)

cfgs = [cfg_256, cfg_1024, cfg_hier]

In [11]:
from typing import Literal


# TODO: Run Experiments
def run_experiments(
    cfgs: list[RAGConfig],
    split: Literal["main", "challenge"] = "main",
    build_pipeline: BuildPipeline = None,
    show_leaderbrd=True,
):
    for cfg in cfgs:
        run_retrieval(cfg, split, cfg.name, build_pipeline)

    if show_leaderbrd:
        show_leaderboard("retrieval", split)

In [25]:
run_experiments(cfgs, split="main")

LLM is explicitly disabled. Using MockLLM.
Collection exist: True


Retr baseline_256: 100%|██████████| 200/200 [00:28<00:00,  7.06it/s]


Embeddings have been explicitly disabled. Using MockEmbedding.
LLM is explicitly disabled. Using MockLLM.
Collection exist: True


Retr baseline_1024: 100%|██████████| 200/200 [00:19<00:00, 10.01it/s]


Embeddings have been explicitly disabled. Using MockEmbedding.
LLM is explicitly disabled. Using MockLLM.
Collection exist: True


Retr baseline_hier: 100%|██████████| 200/200 [00:49<00:00,  4.02it/s]


Embeddings have been explicitly disabled. Using MockEmbedding.
True

🏆 LEADERBOARD [RETRIEVAL | MAIN] 🏆


,run_name,ndcg@k,recall@k,ref_free_mean_score,ref_free_redundancy,latency_s
5,baseline_hier,0.690872,0.755583,0.620421,0.0,49.269500
6,baseline_256,0.679200,0.765750,0.582125,0.0,29.909645
4,baseline_1024,0.675380,0.763250,0.571740,0.0,19.224146
0,baseline,0.673754,0.763250,0.572684,0.0,19.476203


Иерархический чанкинг показал лучший результат по метрикам, но при этом худшую латенси.
Так же видно, что чем меньше чанки, тем лучше результат, что логично, потому что эмбединги более точные

## Task 2: Optimization & Advanced Retrieval (3 балла)

В этом блоке мы реализуем продвинутые техники для улучшения качества и эффективности.

### 2.1 Matryoshka Embeddings (Optimization)
Модель `Qwen/Qwen3-Embedding` поддерживает **Matryoshka Representation Learning (MRL)**. Это позволяет усекать размерность векторов (например, с 1024 до 512) с минимальной потерей качества, что экономит память и ускоряет поиск.
**Задание:** Измените `truncate_dim` в `EmbeddingConfig` (например, на 512) и проверьте влияние на метрики.

### 2.2 Advanced Retrieval (Hybrid + Rerank)
Вам необходимо модифицировать `build_rag_pipeline` (или создать `build_advanced_pipeline`), добавив следующую логику:
1.  **Hybrid Search:** Если `cfg.retrieval.mode == 'hybrid'`, создайте `BM25Retriever` и объедините его с векторным через `QueryFusionRetriever` (алгоритм Reciprocal Rank Fusion).
2.  **Reranking:** Если `cfg.retrieval.use_reranker == True`, добавьте `SentenceTransformerRerank` в список `node_postprocessors` для `QueryEngine`. Реренкер должен принимать на вход топ-K кандидатов из этапа 1 (`overfetch_k`) и оставлять лучшие N (`rerank_top_n`).

**Concept:** Funnel Architecture (Broad Search -> Narrow Filtering).

In [ ]:
# TODO: Upgrade Pipeline Logic (Hybrid, Reranker)
# def build_rag_pipeline(...):
#     ...

# TODO: Run Experiments
# cfg_matryoshka = ...
# cfg_hybrid_rerank = ...
# run_retrieval(cfg_hybrid_rerank, "main", "advanced_full")
# show_leaderboard("retrieval", "main")

In [11]:
cfg_matryoshka = RAGConfig(
    name="matryoshka_512",
    chunking=ChunkingConfig(chunk_size=512, chunk_overlap=50),
    retrieval=RetrievalConfig(top_k=5, overfetch_k=15),
    embedding=EmbeddingConfig(truncate_dim=512),
    qdrant_collection="scifact_matryoshka_512",
    # recreate_collection=True,
)

run_experiments([cfg_matryoshka], show_leaderbrd=False)

c:\Users\Gulfik\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\module.py:1357: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:35.)
  return t.to(


LLM is explicitly disabled. Using MockLLM.


C:\Temp\ipykernel_13852\3636053343.py:11: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <scifact_base_hier_512_128> contains 40127 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  _QDRANT_CLIENT = QdrantClient(path=QDRANT_PATH)


Collection exist: False


Generating embeddings: 100%|██████████| 2048/2048 [02:17<00:00, 14.94it/s]
c:\Users\Gulfik\AppData\Local\Programs\Python\Python312\Lib\site-packages\llama_index\vector_stores\qdrant\base.py:852: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self._client.create_payload_index(
Retr matryoshka_512: 100%|██████████| 200/200 [00:16<00:00, 11.86it/s]


Embeddings have been explicitly disabled. Using MockEmbedding.


In [17]:
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.schema import TextNode
from llama_index.core.query_engine import RetrieverQueryEngine


def build_advanced_pipeline(
    cfg: RAGConfig, df_corpus: pd.DataFrame, load_llm: bool = True
):
    # 1. Setup Models
    Settings.embed_model = get_embedder(cfg.embedding)
    Settings.llm = get_llm(cfg.llm)
    # if load_llm:
    #     Settings.llm = get_llm(cfg.llm)
    # else:
    #     Settings.llm = None

    documents = df_to_documents(df_corpus)

    # 2. Vector Store Setup (TODO)
    client = get_qdrant_client()
    vector_store = QdrantVectorStore(
        collection_name=cfg.qdrant_collection, client=client
    )
    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    # 3. Indexing Logic (TODO)
    # Check if collection exists.
    # If yes & not recreate -> load index.
    # Else -> create documents, setup splitter, build index.
    # print(client.collection_exists(collection_name=cfg.qdrant_collection))
    is_collection_exist = client.collection_exists(
        collection_name=cfg.qdrant_collection
    )
    print(f"Collection exist: {is_collection_exist}")

    if is_collection_exist and not cfg.recreate_collection:
        index = VectorStoreIndex.from_vector_store(
            vector_store=vector_store,
            storage_context=storage_context,
            show_progress=True,
        )
    else:
        if cfg.chunking.use_hierarchical:
            parser = HierarchicalNodeParser.from_defaults(
                chunk_sizes=[cfg.chunking.chunk_size, cfg.chunking.child_chunk_size],
                chunk_overlap=cfg.chunking.chunk_overlap,
            )
            transformations = [parser]
        else:
            splitter = SentenceSplitter(
                chunk_size=cfg.chunking.chunk_size,
                chunk_overlap=cfg.chunking.chunk_overlap,
            )
            transformations = [splitter]

        client.delete_collection(collection_name=cfg.qdrant_collection)
        splitter = SentenceSplitter(
            chunk_size=cfg.chunking.chunk_size,
            chunk_overlap=cfg.chunking.chunk_overlap,
        )
        index = VectorStoreIndex.from_documents(
            documents=documents,
            storage_context=storage_context,
            transformations=transformations,
            show_progress=True,
        )

    # 4. Retrieval Construction (TODO)
    # retriever = ... (use cfg.retrieval.overfetch_k)
    dense_retriever = index.as_retriever(similarity_top_k=cfg.retrieval.overfetch_k)

    if cfg.retrieval.mode == "hybrid":
        splitter = SentenceSplitter(
            chunk_size=cfg.chunking.chunk_size,
            chunk_overlap=cfg.chunking.chunk_overlap,
        )

        nodes = []
        for document in documents:
            for chunk in splitter.split_text(document.text):
                nodes.append(TextNode(text=chunk, metadata=document.metadata))

        bm25_retriever = BM25Retriever.from_defaults(
            nodes=nodes,  # возможно правильнее было сделать через DocumentStore
            similarity_top_k=cfg.retrieval.overfetch_k,
        )

        retriever = QueryFusionRetriever(
            retrievers=[dense_retriever, bm25_retriever],
            similarity_top_k=cfg.retrieval.overfetch_k,
            num_queries=1,
            use_async=False,
            mode="reciprocal_rerank",
        )
    else:
        retriever = dense_retriever

    reranker = None
    if cfg.retrieval.use_reranker:
        reranker = SentenceTransformerRerank(
            model=cfg.rerank_model,
            top_n=cfg.retrieval.rerank_top_n,
        )

    # query_engine = ...
    query_engine = RetrieverQueryEngine.from_args(
        retriever=retriever,
        llm=Settings.llm,
        node_postprocessors=[reranker] if reranker is not None else None,
    )

    return index, retriever, query_engine

In [12]:
cfg_hybrid_rerank = RAGConfig(
    name="hybrid_rerank",
    chunking=ChunkingConfig(chunk_overlap=50),
    retrieval=RetrievalConfig(
        top_k=5,
        overfetch_k=30,  # сделал чуть побольше
        mode="hybrid",
        use_reranker=True,
        rerank_top_n=5,
    ),
    qdrant_collection="scifact_hybrid_512",
    recreate_collection=True,
)

run_experiments([cfg_hybrid_rerank], build_pipeline=build_advanced_pipeline)

c:\Users\Gulfik\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\module.py:1357: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:35.)
  return t.to(


LLM is explicitly disabled. Using MockLLM.


C:\Temp\ipykernel_25108\3636053343.py:11: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <scifact_base_hier_512_128> contains 40127 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  _QDRANT_CLIENT = QdrantClient(path=QDRANT_PATH)


Collection exist: False


Generating embeddings: 100%|██████████| 2048/2048 [02:16<00:00, 14.99it/s]
c:\Users\Gulfik\AppData\Local\Programs\Python\Python312\Lib\site-packages\llama_index\vector_stores\qdrant\base.py:852: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self._client.create_payload_index(
Generating embeddings: 100%|██████████| 1531/1531 [01:44<00:00, 14.69it/s]
Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-Reranker-0.6B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Retr hybrid_rerank: 100%|██████████| 200/200 [00:20<00:00,  9.65it/s]


Embeddings have been explicitly disabled. Using MockEmbedding.

🏆 LEADERBOARD [RETRIEVAL | MAIN] 🏆


,run_name,ndcg@k,recall@k,ref_free_mean_score,ref_free_redundancy,latency_s
0,baseline,0.673754,0.763250,0.572684,0.0,19.476203
1,baseline_1024,0.675380,0.763250,0.571740,0.0,19.989490
2,baseline_256,0.679200,0.765750,0.582125,0.0,28.620433
3,baseline_hier,0.690872,0.755583,0.620421,0.0,52.175176
4,hybrid_rerank,0.650781,0.737083,0.029738,0.0,20.719649
5,matryoshka_512,0.655680,0.754083,0.604436,0.0,16.863469


## Task 3: Generator Tuning (Prompt Engineering) (1 балл)

Качество ретривала (nDCG) мы оптимизировали. Теперь фокус на Генераторе.

**Задание:**
Модифицируйте `prompt_template` в `LLMConfig`. Цель — максимизировать метрики **Faithfulness** и **Answer Relevancy**.
Рекомендуемые техники:
*   **Role Prompting:** ("You are an expert scientist...")
*   **Constraint Enforcement:** ("Answer ONLY based on the context. If unsure, say 'I don't know'.")
*   **Chain-of-Thought:** ("Let's analyze the context step by step...")

In [ ]:
# TODO: Create Prompt Config
# cfg_prompt = ...
# run_e2e(cfg_prompt, "main", "prompt_tuned")
# show_leaderboard("e2e", "main")

In [12]:
PROMPT = """You are an expert scientific assistant.

Use ONLY the information in the provided context to answer the query.
Do NOT use outside knowledge. Do NOT guess.
If the context does not contain enough information to answer, reply exactly: "I don't know."

Follow this procedure silently:
1) Identify the sentences in the context that are relevant to the query.
2) Form the answer using only those sentences.

Context:
{context_str}

Query:
{query_str}

Answer (one short paragraph, grounded in the context):
"""

In [13]:
cfg_prompt = RAGConfig(
    name="prompt_tuned",
    chunking=ChunkingConfig(chunk_size=512, chunk_overlap=50),
    retrieval=RetrievalConfig(top_k=5, overfetch_k=15),
    llm=LLMConfig(prompt_template=PROMPT),
    qdrant_collection="scifact_prompt_tuned_512",
    recreate_collection=True,
)
run_experiments([cfg_prompt])

c:\Users\Gulfik\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\module.py:1357: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:35.)
  return t.to(


LLM is explicitly disabled. Using MockLLM.


C:\Temp\ipykernel_22628\3636053343.py:11: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <scifact_base_hier_512_128> contains 40127 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  _QDRANT_CLIENT = QdrantClient(path=QDRANT_PATH)


Collection exist: False


Generating embeddings: 100%|██████████| 2048/2048 [02:18<00:00, 14.84it/s]
c:\Users\Gulfik\AppData\Local\Programs\Python\Python312\Lib\site-packages\llama_index\vector_stores\qdrant\base.py:852: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self._client.create_payload_index(
Retr prompt_tuned: 100%|██████████| 200/200 [00:19<00:00, 10.23it/s]


Embeddings have been explicitly disabled. Using MockEmbedding.

🏆 LEADERBOARD [RETRIEVAL | MAIN] 🏆


,run_name,ndcg@k,recall@k,ref_free_mean_score,ref_free_redundancy,latency_s
0,baseline,0.673754,0.763250,0.572684,0.0,19.476203
1,baseline_1024,0.675380,0.763250,0.571740,0.0,19.989490
2,baseline_256,0.679200,0.765750,0.582125,0.0,28.620433
3,baseline_hier,0.690872,0.755583,0.620421,0.0,52.175176
4,hybrid_rerank,0.650781,0.737083,0.029738,0.0,20.719649
5,matryoshka_512,0.655680,0.754083,0.604436,0.0,16.863469
6,prompt_tuned,0.673754,0.763250,0.572684,0.0,19.549834


Prompt tuned версия вообще не отличается от бейзлайна по метрикам, видимо, исходная квен и так достаточно умная, поэтому промпт не помог. Либо промпт не самый лучший.

## 4. Final Challenge

Выберите одну лучшую конфигурацию по совокупности метрик. Запустите её на отложенной выборке **Challenge Split**.
Сравните результаты с Baseline на этом же сплите.

Попробую 2 конфига для сравнения: `baseline_256` и `baseline_hier`

In [18]:
baseline_cfg = RAGConfig(
    name="baseline",
    chunking=ChunkingConfig(chunk_size=512, chunk_overlap=50),
    retrieval=RetrievalConfig(top_k=5, overfetch_k=15),
    qdrant_collection="scifact_base_512",
    # recreate_collection=True,
)

cfg_256 = RAGConfig(
    name="baseline_256",
    chunking=ChunkingConfig(chunk_size=256, chunk_overlap=50),
    retrieval=RetrievalConfig(top_k=5, overfetch_k=15),
    qdrant_collection="scifact_base_256",
    # recreate_collection=True,
)

cfg_hier = RAGConfig(
    name="baseline_hier",
    chunking=ChunkingConfig(use_hierarchical=True),
    retrieval=RetrievalConfig(top_k=5, overfetch_k=15),
    qdrant_collection="scifact_base_hier_512_128",
    # recreate_collection=True,
)


best_cfgs = [baseline_cfg, cfg_256, cfg_hier]

In [19]:
# TODO: Final Evaluation
for best_cfg in best_cfgs:
    run_retrieval(best_cfg, "challenge", f"final_submission_{best_cfg.name}")
    run_e2e(best_cfg, "challenge", f"final_submission_{best_cfg.name}")

print("FINAL RESULTS:")
show_leaderboard("retrieval", "challenge")
show_leaderboard("e2e", "challenge")

c:\Users\Gulfik\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\module.py:1357: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:35.)
  return t.to(
Loading checkpoint shards: 100%|██████████| 3/3 [00:05<00:00,  1.85s/it]
C:\Temp\ipykernel_18592\3636053343.py:11: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <scifact_base_hier_512_128> contains 40127 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  _QDRANT_CLIENT = QdrantClient(path=QDRANT_PATH)


Collection exist: True


Retr final_submission_baseline: 100%|██████████| 50/50 [00:05<00:00,  9.55it/s]


Embeddings have been explicitly disabled. Using MockEmbedding.
Collection exist: True


Retrieving Contexts: 100%|██████████| 10/10 [00:01<00:00,  5.28it/s]


...

Judging final_submission_baseline_hier: 100%|██████████| 10/10 [54:22<00:00, 326.28s/it]

FINAL RESULTS:

🏆 LEADERBOARD [RETRIEVAL | CHALLENGE] 🏆


,run_name,ndcg@k,recall@k,ref_free_mean_score,ref_free_redundancy,latency_s
0,final_submission_baseline,0.640820,0.763333,0.555685,0.0,5.289645
1,final_submission_baseline_256,0.625275,0.723333,0.560919,0.0,10.760666
2,final_submission_baseline_hier,0.651485,0.763333,0.589463,0.0,12.426787



🏆 LEADERBOARD [E2E | CHALLENGE] 🏆


,run_name,faithfulness,answer_relevancy,latency_s
0,final_submission_baseline,0.7,0.9,4906.678229
1,final_submission_baseline_256,0.8,1.0,5132.329493
2,final_submission_baseline_hier,0.7,0.9,2662.631020


Не смотря а то, что на трейне hierarchical показал лучший результат, на тесте лучшим оказался baseline с chunk size = 256, хотя и латенси в 2 раза больше